In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Data Reading

In [0]:
#read codes in batch (Process a finite set of data at once)
"""
df = spark.read.format("parquet") \
    .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/data/DimUser")
"""

'\ndf = spark.read.format("parquet")     .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/data/DimUser")\n'

In [0]:
#display(df)

#error->
"""
[UNAUTHORIZED_ACCESS] Unauthorized access:
PERMISSION_DENIED: User does not have READ FILES on External Location 'bronze_external_location'. SQLSTATE: 42501
"""

#solution:
"""
UI steps
1.Go to Catalog in the left sidebar.
2.Open External Locations.

Select:
bronze_external_location

3.Click Permissions.
4.Click Grant.
In Principal, select your user or group.

5.Under privileges, select:

READ FILES

6.Click Grant.

<--OR-->
GRANT READ FILES
ON EXTERNAL LOCATION bronze_external_location
TO `your_user_or_group`;
"""

'\nUI steps\n1.Go to Catalog in the left sidebar.\n2.Open External Locations.\n\nSelect:\nbronze_external_location\n\n3.Click Permissions.\n4.Click Grant.\nIn Principal, select your user or group.\n\n5.Under privileges, select:\n\nREAD FILES\n\n6.Click Grant.\n\n<--OR-->\nGRANT READ FILES\nON EXTERNAL LOCATION bronze_external_location\nTO `your_user_or_group`;\n'

In [0]:
# We want Incremental read (i.e Process only the data that is new or changed since the previous run) using 'readStream'

In [0]:
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation", f"abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimUser") \
    .load(f"abfss://bronze@adlspotifyde.dfs.core.windows.net/DimUser/")


In [0]:
dbutils.fs.ls("abfss://bronze@adlspotifyde.dfs.core.windows.net/")

[FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimArtist/', name='DimArtist/', size=0, modificationTime=1789139755000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimArtist_cdc/', name='DimArtist_cdc/', size=0, modificationTime=1789125741000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimDate/', name='DimDate/', size=0, modificationTime=1789139890000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimDate_cdc/', name='DimDate_cdc/', size=0, modificationTime=1789125757000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack/', name='DimTrack/', size=0, modificationTime=1789139825000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimTrack_cdc/', name='DimTrack_cdc/', size=0, modificationTime=1789125747000),
 FileInfo(path='abfss://bronze@adlspotifyde.dfs.core.windows.net/DimUser/', name='DimUser/', size=0, modificationTime=1789139678000),
 FileInfo(path='abfss://br

In [0]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
df = df.withColumn("first_name",split(col("user_name")," ")[0])
df = df.withColumn("last_name",split(col("user_name")," ")[1])


## Python Utilities

In [0]:
import os
import sys

#as Utils.py at </Workspace/SpotifyDE/Utils.py>

Utils_directory = os.path.join(os.getcwd(),"..")
sys.path.append(Utils_directory)

print(Utils_directory)


/Workspace/SpotifyDE/Silver/..


In [0]:
# from Helper.Utils import reusable

# df_user_obj = reusable()

# df = df_user_obj.dropColumns(df,["_rescued_data"])


In [0]:
#OR 
df = df.drop("_rescued_data")

In [0]:
df = df.dropDuplicates(['user_id'])

In [0]:
#df = df.withColumn("updated_at", to_date(col("updated_at")))

In [0]:
# display(df,checkpointLocation= "abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimUser")


Checkpointing to abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimUser


In [0]:
is_currentdf_group = df.groupBy("subscription_type").agg(count("subscription_type").alias("subscription_total"))

## Data Writing

In [0]:
"""
df.write.format("delta") \
    .option("mode","append") \
    .save("abfss://silver@adlspotifyde.dfs.core.windows.net/user")
"""

    #write directly to a storage path, doesn't create unitycatalog

'\ndf.write.format("delta")     .option("mode","append")     .save("abfss://silver@adlspotifyde.dfs.core.windows.net/user")\n'

In [0]:
df.writeStream.format("delta") \
    .option("outputMode","append") \
    .option("checkpointLocation",f"abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/checkpoint_DimUser") \
    .option("path",f"abfss://silver@adlspotifyde.dfs.core.windows.net/DimUser/data") \
    .trigger(once=True) \
    .start()


In [0]:
dbutils.fs.ls("abfss://silver@adlspotifyde.dfs.core.windows.net/")

[FileInfo(path='abfss://silver@adlspotifyde.dfs.core.windows.net/checkpoint/', name='checkpoint/', size=0, modificationTime=0)]

## Creating Tables

The reason we create a table is that it gives Databricks a metadata layer and a convenient SQL/data-management interface over the files.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS spotifyde_catalog

In [0]:
%sql
-- DROP SCHEMA spotifyde_catalog.silver CASCADE;
-- DROP SCHEMA spotifyde_catalog.gold CASCADE;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7189630266115654>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DROP SCHEMA spotifyde_catalog.silver CASCADE;\nDROP SCHEMA spotifyde_catalog.gold CASCADE;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseExc

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS spotifyde_catalog.silver

In [0]:
display(
    spark.read
        .format("parquet")
        .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/DimUser/")
)

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-7189630266115656>, line 1
----> 1 display(
      2     spark.read
      3         .format("parquet")
      4         .load("abfss://bronze@adlspotifyde.dfs.core.windows.net/DimUser/")
      5 )

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:111, in Display.display_connect_table(self, df, **kwargs)
    109 def display_connect_table(self, df: ConnectDataFrame, **kwargs):
    110     config = self._get_table_display_config()
--> 111     if df.isStreaming:
    112         sel

In [0]:
display(
    spark.read
        .format("delta")
        .load("abfss://silver@adlspotifyde.dfs.core.windows.net/DimUser/data/")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7189630266115655>, line 1
----> 1 display(
      2     spark.read
      3         .format("delta")
      4         .load("abfss://silver@adlspotifyde.dfs.core.windows.net/DimUser/data/")
      5 )

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:115, in Display.display_connect_table(self, df, **kwargs)
    112     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listener,
    113                                                **kwargs)
    114 else:
-->

In [0]:
%sql
--this is an externally located table
CREATE TABLE IF NOT EXISTS spotifyde_catalog.silver.user
USING DELTA LOCATION 'abfss://silver@adlspotifyde.dfs.core.windows.net/DimUser/data/'

--Create/register a table named 'user' whose data is located at this ADLS path.

-- MANAGED LOCATION is primarily used for catalog/schema managed storage, whereas LOCATION is used to specify the storage location of a table.

In [0]:
%sql
SELECT * FROM
spotifyde_catalog.silver.user

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7065943542505569>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * FROM\nspotifyde_catalog.silver.user\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_

In [0]:
"""
df.write.format("delta") \
    .option("mode","append") \
    .saveAsTable("spotifyde_catalog.silver.user")
"""

#saveAsTable --> if catalog/schema exists, 
# ├── Creates table if applicable
#      ├── Registers/uses it in Unity Catalog
#      └── Writes DataFrame data
#                   ↓
#                  ADLS